In [ ]:
# deep_pain_classification_raw_eeg.py
import os
import numpy as np
import pandas as pd
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import random
import mne

# ===============================================
# CONFIGURATION
# ===============================================
DATA_DIR = Path("Segment-Joined")
SFREQ = 250
WIN_SECS = 5
WIN_SAMPLES = WIN_SECS * SFREQ
STEP = WIN_SAMPLES // 2
N_CHANNELS = 24
EPOCHS = 30
BATCH_SIZE = 32
RANDOM_STATE = 42

# Pain score map
PAIN_SCORE = {
    0: 7, 1: 4, 2: 3, 3: 8, 4: 5, 5: 2, 6: 7, 7: 3, 8: 4, 9: 9,
    10: 3, 11: 6, 13: 3, 14: 3, 15: 8, 16: 5, 18: 5, 19: 8, 20: 7,
    21: 6, 22: 7, 23: 6, 24: 9, 25: 8, 26: 0, 27: 1, 30: 3, 31: 9,
    33: 6, 35: 1, 37: 0, 38: 7, 39: 8, 40: 4, 41: 7, 43: 6
}

def pain_label(score):
    if score in (1, 2, 3, 4):
        return "low"
    elif score in (5, 6):
        return "mid"
    elif score in (7, 8, 9):
        return "high"
    else:
        return None

# ===============================================
# LOAD AND WINDOWING
# ===============================================
def load_eeg_data(file_path):
    """Load EEG data from .npy or .fif"""
    if file_path.suffix == ".npy":
        arr = np.load(file_path)
        if arr.ndim == 3: arr = arr.squeeze()
        if arr.shape[0] != N_CHANNELS:
            arr = arr.reshape(N_CHANNELS, -1)
        return arr
    elif file_path.suffix == ".fif":
        raw = mne.io.read_raw_fif(str(file_path), preload=True, verbose="ERROR")
        return raw.get_data()
    else:
        return None

def create_windows(data, win_size, step):
    """Split EEG into overlapping windows"""
    n = data.shape[1]
    windows = []
    for start in range(0, n - win_size + 1, step):
        end = start + win_size
        segment = data[:, start:end]  # shape (24, 1250)
        windows.append(segment)
    return np.stack(windows)

# ===============================================
# PREPARE DATASET
# ===============================================
X, y, subj_ids = [], [], []
files = sorted(DATA_DIR.glob("ID*_combine.npy"))
print(f"Found {len(files)} EEG files.")

for f in files:
    subj_id = int(''.join([c for c in f.stem if c.isdigit()]))
    score = PAIN_SCORE.get(subj_id)
    label = pain_label(score)
    if label is None:
        continue
    data = load_eeg_data(f)
    if data is None or data.shape[1] < WIN_SAMPLES:
        continue
    segments = create_windows(data, WIN_SAMPLES, STEP)
    X.append(segments)
    y.extend([label] * len(segments))
    subj_ids.extend([subj_id] * len(segments))
    print(f"ID{subj_id} -> {label} | windows={len(segments)}")

X = np.concatenate(X, axis=0)
X = np.transpose(X, (0, 2, 1))  # (samples, time, channels)
print(f"Final X shape: {X.shape}")

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
n_classes = len(le.classes_)
print(f"Classes: {le.classes_}")

# ===============================================
# SUBJECT-LEVEL SPLIT
# ===============================================
np.random.seed(RANDOM_STATE)
unique_subjects = list(set(subj_ids))
random.shuffle(unique_subjects)

train_subj, test_subj = train_test_split(
    unique_subjects, test_size=0.2, random_state=RANDOM_STATE, stratify=[pain_label(PAIN_SCORE[s]) for s in unique_subjects if pain_label(PAIN_SCORE[s])])

train_idx = [i for i, sid in enumerate(subj_ids) if sid in train_subj]
test_idx = [i for i, sid in enumerate(subj_ids) if sid in test_subj]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
print(f"Train: {len(X_train)} samples, Test: {len(X_test)} samples")

# ===============================================
# BUILD MODEL (Conv + BiLSTM)
# ===============================================
def build_conv_lstm_model(input_shape, n_classes):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, 7, activation='relu', padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=False))(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_conv_lstm_model((WIN_SAMPLES, N_CHANNELS), n_classes)
model.summary()

# ===============================================
# TRAIN
# ===============================================
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# ===============================================
# PLOTS
# ===============================================
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title("Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig("training_curves.png")
plt.show()

# ===============================================
# EVALUATION
# ===============================================
y_pred = np.argmax(model.predict(X_test), axis=1)

acc = np.mean(y_pred == y_test)
print(f"✅ Test Accuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.show()
